# SSL Watermark Benchmark — Google Colab
**Post-hoc, self-supervised (Fernandez et al. 2021).**  
Embed = per-image Adam optimisation on frozen DINO features.  
Detect = batched backbone forward pass.  
Runs 14 WAVES attacks, reports PSNR / SSIM / BitAcc / AUROC / TPR@1% FPR.

### Setup
1. Runtime → **GPU** (T4 or better)
2. Clones [github.com/ademladhari/wavess](https://github.com/ademladhari/wavess) (+ `ssl` submodule)
3. Put COCO images on Drive or use the optional COCO download cell

In [ ]:
# ── 1. Clone wavess + mount Drive (images + results) ─────────────────────────
import os, sys, subprocess

REPO_URL   = 'https://github.com/ademladhari/wavess.git'
WAVES_ROOT = '/content/wavess'

if not os.path.isdir(f'{WAVES_ROOT}/ssl'):
    print('Cloning wavess (with ssl submodule)…')
    subprocess.run(['git', 'clone', '--depth', '1', '--recurse-submodules', REPO_URL, WAVES_ROOT], check=True)
else:
    print('Repo already present:', WAVES_ROOT)

SSL_ROOT = f'{WAVES_ROOT}/ssl'
for p in [WAVES_ROOT, SSL_ROOT, f'{SSL_ROOT}/src']:
    if p not in sys.path:
        sys.path.insert(0, p)

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

SAVE_TO_DRIVE = True
DRIVE_OUTPUT  = '/content/drive/MyDrive/wavess_results/ssl'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

_ba = f'{WAVES_ROOT}/benchmark_attacks.py'
if not os.path.isfile(_ba):
    raise FileNotFoundError(
        f'Missing {_ba} — run: git push origin main  (from D:\\waves on your PC)'
    )
print('Repo OK:', WAVES_ROOT, '| benchmark_attacks.py found')

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────────────
!pip install -q scikit-image scikit-learn tqdm PyYAML
# torchvision, torch already present in Colab

In [ ]:
# ── 3. (Optional) Download COCO val images if not on Drive ───────────────────
# Skip this cell if your images are already in IMAGES_DIR below.

# import urllib.request, zipfile, pathlib
# COCO_DIR = '/content/coco_val'
# pathlib.Path(COCO_DIR).mkdir(exist_ok=True)
# url = 'http://images.cocodataset.org/zips/val2017.zip'
# print('Downloading COCO val2017 (~800 MB)…')
# urllib.request.urlretrieve(url, '/content/val2017.zip')
# with zipfile.ZipFile('/content/val2017.zip') as z:
#     z.extractall('/content')
# print('Done.')

In [ ]:
# ── 4. Config — edit here ────────────────────────────────────────────────────
IMAGES_DIR    = '/content/drive/MyDrive/data/coco100k/val'  # or /content/val2017 after COCO download
N_IMAGES      = 100
IMAGE_SIZE    = 512
MODE          = 'multi_bit'          # 'multi_bit' or 'zero_bit'
EMBED_BATCH   = 1                    # embed is per-image Adam opt (always 1)
DETECT_BATCH  = 16                   # backbone inference batch size
OUTPUT_DIR    = f'{WAVES_ROOT}/ssl/outputs_benchmark_colab'
SEED          = 42
DEVICE        = 'cuda'

# Checkpoint paths (must exist on Drive)
WHITENING_PT  = f'{SSL_ROOT}/checkpoints/whitening.pt'
KEY_FILE      = f'{SSL_ROOT}/checkpoints/key_multi.npy'  # or key_zero.npy

In [ ]:
# ── 5. Imports ────────────────────────────────────────────────────────────────
import csv, json
from pathlib import Path

import numpy as np
import torch
import yaml
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from sklearn import metrics as sk_metrics
from tqdm.auto import tqdm

# SSL modules
sys.path.insert(0, SSL_ROOT)
from src.backbone import Backbone, PCAWhitening
from src.constraints import psnr as ssl_psnr
from src.embed import EmbedConfig, embed_multi_bit, embed_zero_bit
from src.detect import decode_multi_bit, detect_zero_bit
from src.keys import load_key

# Shared attacks
from benchmark_attacks import ATTACKS, apply_attack_rgb

device = torch.device(DEVICE)
print('Imports OK  |  GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')

In [ ]:
# ── 6. Helpers ────────────────────────────────────────────────────────────────
_IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

def list_images(root, n):
    paths = sorted(p for p in Path(root).iterdir()
                   if p.is_file() and p.suffix.lower() in _IMG_EXTS)
    if len(paths) < n:
        raise FileNotFoundError(f'Need {n} images, found {len(paths)} in {root}')
    return paths[:n]

def pil_to_tensor(im):
    arr = np.asarray(im.convert('RGB'), dtype=np.float32) / 255.0
    return torch.from_numpy(arr).permute(2, 0, 1)

def tensor_to_pil(t):
    if t.ndim == 4: t = t[0]
    arr = t.detach().clamp(0,1).mul(255).round().to(torch.uint8).permute(1,2,0).cpu().numpy()
    return Image.fromarray(arr, 'RGB')

def load_rgb_tensor(path, size):
    with Image.open(path) as im:
        im = im.convert('RGB')
        w, h = im.size; s = min(w, h)
        l, t = (w-s)//2, (h-s)//2
        im = im.crop((l, t, l+s, t+s))
        if im.size != (size, size):
            im = im.resize((size, size), Image.Resampling.LANCZOS)
    return pil_to_tensor(im).unsqueeze(0).to(device)

def psnr_ssim_pil(ref, atk):
    b = atk.resize(ref.size, Image.Resampling.LANCZOS) if atk.size != ref.size else atk
    a = np.asarray(ref, np.float64); b = np.asarray(b, np.float64)
    return (float(peak_signal_noise_ratio(a, b, data_range=255.0)),
            float(structural_similarity(a, b, channel_axis=2, data_range=255.0)))

def ba_multi(dec, msg):
    return float((dec.sign() == msg.sign()).float().mean().item())

def score_zero(s):
    return float(torch.sigmoid(s[0]).item())

def auroc_tpr(pos, neg, fpr=0.01):
    y = np.concatenate([np.zeros(neg.size, np.int32), np.ones(pos.size, np.int32)])
    s = np.concatenate([neg, pos])
    auc = float(sk_metrics.roc_auc_score(y, s))
    fp, tp, _ = sk_metrics.roc_curve(y, s, pos_label=1)
    idx = np.where(fp < fpr)[0]
    return auc, float(tp[idx[-1]]) if idx.size else float(tp[0])

print('Helpers defined.')

In [ ]:
# ── 7. Load backbone + config ─────────────────────────────────────────────────
with open(f'{SSL_ROOT}/configs/{MODE}.yaml') as f:
    cfg = yaml.safe_load(f)

whitening = PCAWhitening.load(WHITENING_PT)
backbone  = Backbone(whitening=whitening, feat_dim=int(cfg['feat_dim'])).to(device).eval()
key       = load_key(KEY_FILE).to(device)

embed_cfg = EmbedConfig(
    target_psnr=float(cfg['target_psnr']),
    n_iter=int(cfg['n_iter']),
    lr=float(cfg['lr']),
    lambda_w=float(cfg['lambda_w']),
)
fpr_val  = float(cfg.get('fpr', 1e-6))
margin   = float(cfg.get('margin', 5.0))
n_bits   = int(cfg.get('n_bits', 30)) if MODE == 'multi_bit' else 0
print(f'Backbone loaded  |  mode={MODE}  n_bits={n_bits}  n_iter={embed_cfg.n_iter}')

In [ ]:
# ── 8. Embed watermarks (per-image Adam opt — inherently sequential) ──────────
paths = list_images(IMAGES_DIR, N_IMAGES)
records = []

print(f'Embedding {len(paths)} images (n_iter={embed_cfg.n_iter}, PSNR target={embed_cfg.target_psnr} dB)…')
for i, path in enumerate(tqdm(paths, desc='embed', unit='img')):
    host_t   = load_rgb_tensor(path, IMAGE_SIZE)
    host_pil = tensor_to_pil(host_t)
    if MODE == 'multi_bit':
        gen = torch.Generator().manual_seed(SEED + i)
        msg = (torch.randint(0, 2, (1, n_bits), generator=gen).float() * 2.0 - 1.0).to(device)
        wm_t = embed_multi_bit(backbone, host_t, carriers=key,
                               messages=msg, margin=margin, config=embed_cfg, seed=SEED)
    else:
        msg = None
        wm_t = embed_zero_bit(backbone, host_t, key=key,
                               fpr=fpr_val, config=embed_cfg, seed=SEED)
    records.append({'path': str(path), 'host_pil': host_pil,
                    'wm_pil': tensor_to_pil(wm_t), 'message': msg,
                    'embed_psnr': float(ssl_psnr(wm_t, host_t).mean().item())})

print(f'Mean embed PSNR: {np.mean([r["embed_psnr"] for r in records]):.2f} dB')

In [ ]:
# ── 9. Sanity check + negative scores (batched backbone) ─────────────────────
@torch.no_grad()
def decode_batch(pil_list):
    """Decode a batch of PIL images; returns list of (decoded_tensor, score)."""
    tensors = torch.stack([pil_to_tensor(p) for p in pil_list]).to(device)
    if MODE == 'multi_bit':
        return decode_multi_bit(backbone, tensors, carriers=key), None
    else:
        det, sc = detect_zero_bit(backbone, tensors, key=key, fpr=fpr_val)
        return None, sc

# sanity
if MODE == 'multi_bit':
    dec0, _ = decode_batch([records[0]['wm_pil']])
    print(f'Sanity (image 0, no attack): BitAcc = {ba_multi(dec0[0], records[0]["message"][0]):.3f}')

# negatives in batches
print('Scoring negatives (batched)…')
neg_scores = []
for i in tqdm(range(0, len(records), DETECT_BATCH), desc='negatives'):
    batch = records[i:i+DETECT_BATCH]
    host_pils = [r['host_pil'] for r in batch]
    if MODE == 'multi_bit':
        dec, _ = decode_batch(host_pils)
        for j, r in enumerate(batch):
            neg_scores.append(ba_multi(dec[j], r['message'][0]))
    else:
        _, sc = decode_batch(host_pils)
        neg_scores.extend([score_zero(sc[j:j+1]) for j in range(len(batch))])

neg_arr = np.asarray(neg_scores, np.float64)
print(f'Neg score mean: {neg_arr.mean():.4f}')

In [ ]:
# ── 10. Per-attack evaluation (batched detection) ────────────────────────────
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
rows = []

for spec in ATTACKS:
    psnr_vals, ssim_vals, bit_vals, pos_scores = [], [], [], []

    # apply attack to all images first
    attacked = [apply_attack_rgb(spec.name, r['wm_pil'], seed=i)
                for i, r in enumerate(records)]

    # PSNR/SSIM
    for ref, atk in zip(records, attacked):
        p, s = psnr_ssim_pil(ref['host_pil'], atk)
        psnr_vals.append(p); ssim_vals.append(s)

    # batched decode
    with torch.no_grad():
        for i in range(0, len(attacked), DETECT_BATCH):
            batch_atk  = attacked[i:i+DETECT_BATCH]
            batch_recs = records[i:i+DETECT_BATCH]
            if MODE == 'multi_bit':
                dec, _ = decode_batch(batch_atk)
                for j, r in enumerate(batch_recs):
                    ba = ba_multi(dec[j], r['message'][0])
                    bit_vals.append(ba); pos_scores.append(ba)
            else:
                _, sc = decode_batch(batch_atk)
                for j in range(len(batch_atk)):
                    s_val = score_zero(sc[j:j+1])
                    pos_scores.append(s_val); bit_vals.append(float('nan'))

    pos_arr = np.asarray(pos_scores, np.float64)
    auc, tpr1 = auroc_tpr(pos_arr, neg_arr)
    ba_mean = float(np.nanmean(bit_vals)) if MODE == 'multi_bit' else float('nan')
    row = {'method': 'ssl', 'mode': MODE, 'attack': spec.name,
           'description': spec.description, 'n_images': len(records),
           'PSNR': float(np.mean(psnr_vals)), 'SSIM': float(np.mean(ssim_vals)),
           'bit_accuracy': ba_mean, 'AUROC': auc, 'TPR_at_1pct_FPR': tpr1}
    rows.append(row)
    print(f"  {spec.name:22s}: BitAcc={ba_mean:.3f}  AUROC={auc:.3f}  TPR@1%={tpr1:.3f}")

print('\nAll attacks done.')

In [ ]:
# ── 11. Save results ──────────────────────────────────────────────────────────
csv_path = Path(OUTPUT_DIR) / 'results.csv'
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader(); w.writerows(rows)
with open(Path(OUTPUT_DIR) / 'results.json', 'w') as f:
    json.dump({'method': 'ssl', 'mode': MODE, 'n_images': N_IMAGES,
               'image_size': IMAGE_SIZE, 'attacks': rows}, f, indent=2)

print(f'Saved to {OUTPUT_DIR}')
if SAVE_TO_DRIVE:
    import shutil
    shutil.copytree(OUTPUT_DIR, DRIVE_OUTPUT, dirs_exist_ok=True)
    print(f'Copied to Drive: {DRIVE_OUTPUT}')

import pandas as pd
df = pd.DataFrame(rows)[['attack','PSNR','SSIM','bit_accuracy','AUROC','TPR_at_1pct_FPR']]
df.columns = ['Attack','PSNR','SSIM','BitAcc','AUROC','TPR@1%FPR']
print(df.to_string(index=False))